# 03 · Limpieza de datos
Lee `datos_crudos_2024200482A.csv` (salida del cuaderno 02) y genera `datos_procesados/datos_procesados_2024200482A.csv`, `salidas/reporte_calidad.csv` y `salidas/hash_sha256.txt`.



In [ ]:
# CELDA 0. Conectar Google Drive (donde están los datos crudos de los cuadernos 01 y 02)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Nombres y apellidos: ASTETE ROSALES GRAYCE NICOL
# Código de matrícula: 2024200482A
# Tema N.º 2 del temario: El margen de intermediación financiera en el Perú: spread TAMN-TIPMN y sus determinantes
# Fecha de extracción:2026-09-25
# ============================================================================
# 03_limpieza_datos
# CELDA 1. Librerías, rutas y lectura de los datos crudos
# ============================================================================
import hashlib, logging
from pathlib import Path
import numpy as np
import pandas as pd

CODIGO_MATRICULA = "2024200482A"
RAIZ = Path("/content/drive/MyDrive/Base de datos y código Grayce Nicol Astete Rosales")
DIR_CRUDOS = RAIZ / "datos_crudos"
DIR_PROC = RAIZ / "datos_procesados"
DIR_SAL = RAIZ / "salidas"
DIR_PROC.mkdir(parents=True, exist_ok=True)
DIR_SAL.mkdir(parents=True, exist_ok=True)

MAX_DIAS_RELLENO = 2   # días máximos que se arrastra el último dato de un determinante (ver PASO 3)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S",
                    handlers=[logging.FileHandler(RAIZ / "log_ejecucion.txt", encoding="utf-8"),
                              logging.StreamHandler()], force=True)
log = logging.getLogger()

# Tabla única de datos crudos (salida del cuaderno 02) y registro del scraping
crudo = pd.read_csv(DIR_CRUDOS / f"datos_crudos_{CODIGO_MATRICULA}.csv", sep=";", decimal=",")
avance = pd.read_csv(DIR_CRUDOS / "avance_scraping_sbs.csv", dtype=str)
log.info(f"INICIO limpieza | datos crudos: {crudo.shape[0]} filas x {crudo.shape[1]} columnas")
from google.colab import data_table; data_table.disable_dataframe_formatter()

2026-09-25 04:53:10 | INICIO limpieza | datos crudos: 1304 filas x 8 columnas


In [ ]:
# ============================================================================
# CELDA 2. Diagnóstico inicial: tipos de dato, faltantes y primeras filas
# ============================================================================
diagnostico = pd.DataFrame({"tipo": crudo.dtypes.astype(str),
                            "con_dato": crudo.notna().sum(),
                            "faltantes": crudo.isna().sum()})
display(diagnostico)
display(crudo.head())

,tipo,con_dato,faltantes
id,int64,1304,0
fecha,object,1304,0
tamn_sbs_pct,float64,1303,1
tipmn_sbs_pct,float64,1304,0
tasa_referencia_bcrp_pct,float64,1247,57
tasa_interbancaria_mn_pct,float64,1245,59
riesgo_pais_embig_pbs,float64,1304,0
tipo_cambio_venta_pen_usd,float64,1246,58


,id,fecha,tamn_sbs_pct,tipmn_sbs_pct,tasa_referencia_bcrp_pct,tasa_interbancaria_mn_pct,riesgo_pais_embig_pbs,tipo_cambio_venta_pen_usd
0,1,2021-01-01,12.10,0.98,NaN,NaN,132.0,NaN
1,2,2021-01-04,12.23,0.98,0.25,0.25,131.0,3.626667
2,3,2021-01-05,12.15,0.97,0.25,0.25,130.0,3.633667
3,4,2021-01-06,12.10,0.96,0.25,0.25,128.0,3.626833
4,5,2021-01-07,12.08,0.96,0.25,0.25,128.0,3.623000


In [ ]:
# ============================================================================
# CELDA 3. Limpieza paso a paso
# ============================================================================
df = crudo.copy()

# PASO 1. Tipificación: la fecha como fecha y las variables como números
df["fecha"] = pd.to_datetime(df["fecha"], format="%Y-%m-%d")
VARIABLES = ["tamn_sbs_pct", "tipmn_sbs_pct", "tasa_referencia_bcrp_pct",
             "tasa_interbancaria_mn_pct", "riesgo_pais_embig_pbs", "tipo_cambio_venta_pen_usd"]
for c in VARIABLES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# PASO 2. Validación de fechas de la SBS: en feriados la SBS muestra el dato de otro día.
# Si la fecha que quedó en la página no es la pedida, ese dato se anula (no corresponde a ese día).
avance = avance.drop_duplicates(["columna", "fecha"], keep="last")
avance["coincide"] = (pd.to_datetime(avance["fecha_en_pagina"], format="%d/%m/%Y", errors="coerce")
                      == pd.to_datetime(avance["fecha"]))
anulados = {}
for col in ["tamn_sbs_pct", "tipmn_sbs_pct"]:
    malas = pd.to_datetime(avance.loc[(avance["columna"] == col) & ~avance["coincide"], "fecha"])
    anulados[col] = int(df["fecha"].isin(malas).sum())
    df.loc[df["fecha"].isin(malas), col] = np.nan
log.info(f"PASO 2 | datos SBS anulados por fecha distinta a la pedida: {anulados}")

# PASO 3. Feriados peruanos: ese día no hay mercado interbancario y el BCRP no publica
# la tasa interbancaria ni la de referencia. Esos días se eliminan (no se rellenan).
feriados = df["tasa_interbancaria_mn_pct"].isna() | df["tasa_referencia_bcrp_pct"].isna()
log.info(f"PASO 3 | feriados o días sin mercado eliminados: {int(feriados.sum())}")
df = df[~feriados].copy()

# Huecos sueltos restantes (p. ej., feriados de EE. UU. en el EMBIG): se arrastra
# el último dato publicado como máximo MAX_DIAS_RELLENO días.
determinantes = VARIABLES[2:]
antes = df[determinantes].isna().sum()
df[determinantes] = df[determinantes].ffill(limit=MAX_DIAS_RELLENO)
rellenados = (antes - df[determinantes].isna().sum()).to_dict()
log.info(f"PASO 3 | huecos rellenados con el último valor disponible: {rellenados}")
# PASO 4. Se conservan solo los días con TODAS las variables (sin datos inventados)
filas_antes = len(df)
df = df.dropna(subset=VARIABLES).reset_index(drop=True)
log.info(f"PASO 4 | días eliminados por datos incompletos: {filas_antes - len(df)} | quedan {len(df)}")

# PASO 5. Nombres normalizados y variables derivadas
df = df.rename(columns={"tamn_sbs_pct": "tamn_pct", "tipmn_sbs_pct": "tipmn_pct",
                        "tasa_referencia_bcrp_pct": "tasa_referencia_pct",
                        "tasa_interbancaria_mn_pct": "tasa_interbancaria_pct",
                        "tipo_cambio_venta_pen_usd": "tipo_cambio_pen_usd"})
df["spread_pct"] = df["tamn_pct"] - df["tipmn_pct"]                          # variable dependiente
df["riesgo_pais_embig_pct"] = df["riesgo_pais_embig_pbs"] / 100              # 100 pbs = 1 punto porcentual
df["brecha_interbancaria_pct"] = df["tasa_interbancaria_pct"] - df["tasa_referencia_pct"]  # presión de liquidez
df = df.drop(columns=["riesgo_pais_embig_pbs", "id"])
df.insert(0, "id", range(1, len(df) + 1))           # nuevo id correlativo tras la limpieza
df = df[["id", "fecha", "tamn_pct", "tipmn_pct", "spread_pct", "tasa_referencia_pct",
         "tasa_interbancaria_pct", "brecha_interbancaria_pct", "riesgo_pais_embig_pct",
         "tipo_cambio_pen_usd"]]
display(df.head())

2026-09-25 04:53:10 | PASO 2 | datos SBS anulados por fecha distinta a la pedida: {'tamn_sbs_pct': 0, 'tipmn_sbs_pct': 0}
2026-09-25 04:53:10 | PASO 3 | feriados o días sin mercado eliminados: 59
2026-09-25 04:53:10 | PASO 3 | huecos rellenados con el último valor disponible: {'tasa_referencia_bcrp_pct': 0, 'tasa_interbancaria_mn_pct': 0, 'riesgo_pais_embig_pbs': 0, 'tipo_cambio_venta_pen_usd': 0}
2026-09-25 04:53:10 | PASO 4 | días eliminados por datos incompletos: 0 | quedan 1245


,id,fecha,tamn_pct,tipmn_pct,spread_pct,tasa_referencia_pct,tasa_interbancaria_pct,brecha_interbancaria_pct,riesgo_pais_embig_pct,tipo_cambio_pen_usd
0,1,2021-01-04,12.23,0.98,11.25,0.25,0.25,0.0,1.31,3.626667
1,2,2021-01-05,12.15,0.97,11.18,0.25,0.25,0.0,1.30,3.633667
2,3,2021-01-06,12.10,0.96,11.14,0.25,0.25,0.0,1.28,3.626833
3,4,2021-01-07,12.08,0.96,11.12,0.25,0.25,0.0,1.28,3.623000
4,5,2021-01-08,12.05,0.96,11.09,0.25,0.25,0.0,1.25,3.613833


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
# ============================================================================
# CELDA 4. Control de calidad: valores imposibles y atípicos
# Los atípicos se REPORTAN pero no se eliminan: en finanzas suelen ser episodios reales
# (por ejemplo, el ciclo de alzas de tasas 2022-2023).
# ============================================================================
numericas = df.columns.drop(["id", "fecha"])
filas = []
for c in numericas:
    s = df[c]
    q1, q3 = s.quantile([0.25, 0.75])
    rango = q3 - q1
    atipicos = df.loc[(s < q1 - 3 * rango) | (s > q3 + 3 * rango), "fecha"]
    filas.append({"variable": c, "observaciones": s.notna().sum(), "minimo": s.min(), "maximo": s.max(),
                  "atipicos_3IQR": len(atipicos),
                  "fechas_atipicas": ", ".join(atipicos.dt.strftime("%Y-%m-%d").head(5))})
calidad = pd.DataFrame(filas)
calidad["cumple_1000"] = calidad["observaciones"] >= 1000

# Valores imposibles: tasas negativas o mayores a 100 %, tipo de cambio no positivo
imposibles = ((df[["tamn_pct", "tipmn_pct", "tasa_referencia_pct", "tasa_interbancaria_pct"]] < 0) |
              (df[["tamn_pct", "tipmn_pct", "tasa_referencia_pct", "tasa_interbancaria_pct"]] > 100)).sum().sum()
imposibles += (df["tipo_cambio_pen_usd"] <= 0).sum()
log.info(f"Control de calidad | valores imposibles: {imposibles}")
calidad.to_csv(DIR_SAL / "reporte_calidad.csv", index=False)
display(calidad)

2026-09-25 04:53:10 | Control de calidad | valores imposibles: 0


,variable,observaciones,minimo,maximo,atipicos_3IQR,fechas_atipicas,cumple_1000
0,tamn_pct,1245,10.340000,32.1900,10,"2023-09-04, 2023-09-15, 2023-10-02, 2024-07-17...",True
1,tipmn_pct,1245,0.750000,4.1000,0,,True
2,spread_pct,1245,9.560000,29.8900,12,"2021-12-07, 2021-12-09, 2023-09-04, 2023-09-15...",True
3,tasa_referencia_pct,1245,0.250000,7.7500,0,,True
4,tasa_interbancaria_pct,1245,0.250000,7.9200,0,,True
5,brecha_interbancaria_pct,1245,-0.910000,0.5000,146,"2022-02-01, 2022-02-04, 2022-02-28, 2022-03-01...",True
6,riesgo_pais_embig_pct,1245,1.170000,2.5700,0,,True
7,tipo_cambio_pen_usd,1245,3.359429,4.1375,0,,True


In [ ]:
# ============================================================================
# CELDA 5. Guardado de la base procesada y hash SHA-256
# El hash se copia en el README y se coteja contra este archivo (numeral 2.4.5)
# ============================================================================
df = df.round(6)                     # 6 decimales: evita residuos de la resta en coma flotante
archivo = DIR_PROC / f"datos_procesados_{CODIGO_MATRICULA}.csv"
df.to_csv(archivo, index=False, sep=";", decimal=",", date_format="%Y-%m-%d", encoding="utf-8-sig")
df.to_excel(DIR_PROC / f"datos_procesados_{CODIGO_MATRICULA}.xlsx", index=False)   # copia para revisar

h = hashlib.sha256(archivo.read_bytes()).hexdigest()
(DIR_SAL / "hash_sha256.txt").write_text(f"{h}  {archivo.name}\n", encoding="utf-8")
log.info(f"Base procesada: {archivo.name} | {df.shape[0]} filas x {df.shape[1]} columnas")
log.info(f"SHA-256 {archivo.name}: {h}")
log.info(f"Periodo: {df['fecha'].min():%Y-%m-%d} a {df['fecha'].max():%Y-%m-%d}")
log.info("FIN limpieza")
print("\nCOPIE ESTE HASH EN SU README:\n", h)

2026-09-25 04:53:11 | Base procesada: datos_procesados_2024200482A.csv | 1245 filas x 10 columnas
2026-09-25 04:53:11 | SHA-256 datos_procesados_2024200482A.csv: 0b57bc94ff94f1798cb40c2fd8bb266046345662631505e4917a56de2e1f8448
2026-09-25 04:53:11 | Periodo: 2021-01-04 a 2025-12-31
2026-09-25 04:53:11 | FIN limpieza



COPIE ESTE HASH EN SU README:
 0b57bc94ff94f1798cb40c2fd8bb266046345662631505e4917a56de2e1f8448
